1. Imports + config

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import re

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

RAW_PATH = "/Volumes/workspace/default/gharchive_raw"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.ingestion_file_log"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.bronze_github_events (
    event_id STRING,
    event_type STRING,
    created_at STRING,
    actor_json STRING,
    repo_json STRING,
    org_json STRING,
    payload_json STRING,
    public BOOLEAN,
    raw_json STRING,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.ingestion_file_log (
    source_file STRING,
    source_hour TIMESTAMP,
    file_status STRING,
    discovered_at TIMESTAMP,
    ingestion_started_at TIMESTAMP,
    ingestion_completed_at TIMESTAMP,
    row_count BIGINT,
    error_message STRING
)
USING DELTA
""")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

if not spark.catalog.tableExists(LOG_TABLE):
    spark.sql(f"""
    CREATE TABLE {LOG_TABLE} (
        source_file STRING,
        source_hour TIMESTAMP,
        file_status STRING,
        discovered_at TIMESTAMP,
        ingestion_started_at TIMESTAMP,
        ingestion_completed_at TIMESTAMP,
        row_count BIGINT,
        error_message STRING
    )
    USING DELTA
    """)

2. Discover files

In [0]:
all_files = sorted([
    f.path
    for f in dbutils.fs.ls(RAW_PATH)
    if re.search(r"\d{4}-\d{2}-\d{2}-\d{1,2}\.json\.gz$", f.path)
])

print("Files discovered:", len(all_files))

3. Identify pending files

In [0]:
completed_files = {
    row["source_file"]
    for row in (
        spark.table(LOG_TABLE)
        .filter(F.col("file_status") == "completed")
        .select("source_file")
        .collect()
    )
}

pending_files = [
    f for f in all_files
    if f not in completed_files
]

print("Completed files:", len(completed_files))
print("Files to process:", len(pending_files))

4. helper for source hour

In [0]:
def extract_source_hour(path):
    match = re.search(
        r"(\d{4}-\d{2}-\d{2})-(\d{1,2})\.json\.gz$",
        path
    )

    if not match:
        return None

    return datetime.strptime(
        f"{match.group(1)} {int(match.group(2)):02d}:00:00",
        "%Y-%m-%d %H:%M:%S"
    )

5. if there is no new file, stop

In [0]:
if not pending_files:
    print("No new files to process.")

6. Register pending files

In [0]:
if pending_files:

    log_delta = DeltaTable.forName(spark, LOG_TABLE)

    discovered_df = (
        spark.createDataFrame(
            [
                (f, extract_source_hour(f))
                for f in pending_files
            ],
            ["source_file", "source_hour"]
        )
        .withColumn("file_status", F.lit("processing"))
        .withColumn("discovered_at", F.current_timestamp())
        .withColumn("ingestion_started_at", F.current_timestamp())
        .withColumn("ingestion_completed_at", F.lit(None).cast("timestamp"))
        .withColumn("row_count", F.lit(None).cast("long"))
        .withColumn("error_message", F.lit(None).cast("string"))
    )

    (
        log_delta.alias("t")
        .merge(
            discovered_df.alias("s"),
            "t.source_file = s.source_file"
        )
        .whenMatchedUpdate(
            condition="t.file_status <> 'completed'",
            set={
                "file_status": "s.file_status",
                "ingestion_started_at": "s.ingestion_started_at",
                "error_message": "s.error_message"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

7. Read only pending files and build Bronze

In [0]:
if pending_files:

    bronze_delta = DeltaTable.forName(spark, BRONZE_TABLE)
    log_delta = DeltaTable.forName(spark, LOG_TABLE)

    for file_path in pending_files:

        print(f"Processing: {file_path}")

        try:
            # Read only this hourly file
            raw_lines = (
                spark.read
                .text(file_path)
                .withColumn(
                    "source_file",
                    F.col("_metadata.file_path")
                )
            )

            bronze_batch = (
                raw_lines
                .select(
                    F.get_json_object("value", "$.id").alias("event_id"),
                    F.get_json_object("value", "$.type").alias("event_type"),
                    F.get_json_object("value", "$.created_at").alias("created_at"),

                    F.get_json_object("value", "$.actor").alias("actor_json"),
                    F.get_json_object("value", "$.repo").alias("repo_json"),
                    F.get_json_object("value", "$.org").alias("org_json"),
                    F.get_json_object("value", "$.payload").alias("payload_json"),

                    F.get_json_object("value", "$.public")
                        .cast("boolean")
                        .alias("public"),

                    F.col("value").alias("raw_json"),
                    F.col("source_file")
                )
                .withColumn(
                    "source_hour",
                    F.lit(extract_source_hour(file_path)).cast("timestamp")
                )
                .withColumn(
                    "ingested_at",
                    F.current_timestamp()
                )
            )

            bronze_batch = bronze_batch.dropDuplicates(["event_id"])

            row_count = bronze_batch.count()

            # Rerun-safe write
            (
                bronze_delta.alias("t")
                .merge(
                    bronze_batch.alias("s"),
                    "t.event_id = s.event_id"
                )
                .whenNotMatchedInsertAll()
                .execute()
            )

            # Mark this specific file as completed
            completed_df = (
                spark.createDataFrame(
                    [(file_path, row_count)],
                    ["source_file", "row_count"]
                )
                .withColumn("file_status", F.lit("completed"))
                .withColumn(
                    "ingestion_completed_at",
                    F.current_timestamp()
                )
                .withColumn(
                    "error_message",
                    F.lit(None).cast("string")
                )
            )

            (
                log_delta.alias("t")
                .merge(
                    completed_df.alias("s"),
                    "t.source_file = s.source_file"
                )
                .whenMatchedUpdate(
                    set={
                        "file_status": "s.file_status",
                        "row_count": "s.row_count",
                        "ingestion_completed_at":
                            "s.ingestion_completed_at",
                        "error_message": "s.error_message"
                    }
                )
                .execute()
            )

            print(f"Completed: {file_path} | rows: {row_count}")

        except Exception as e:

            error_message = str(e)[:4000]

            failed_df = (
                spark.createDataFrame(
                    [(file_path, error_message)],
                    ["source_file", "error_message"]
                )
                .withColumn("file_status", F.lit("failed"))
            )

            (
                log_delta.alias("t")
                .merge(
                    failed_df.alias("s"),
                    "t.source_file = s.source_file"
                )
                .whenMatchedUpdate(
                    set={
                        "file_status": "s.file_status",
                        "error_message": "s.error_message"
                    }
                )
                .execute()
            )

            print(f"FAILED: {file_path}")
            print(error_message)

8. Verification

In [0]:
print(
    "Bronze rows:",
    spark.table(BRONZE_TABLE).count()
)

print(
    "Duplicate event IDs:",
    spark.table(BRONZE_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    "Logged files:",
    spark.table(LOG_TABLE).count()
)

spark.table(LOG_TABLE) \
    .orderBy("source_hour") \
    .show(truncate=False)